In [8]:
import json
import sqlite3

# ============================================================
# 1. CONECTARSE A LA BASE DE DATOS
# ============================================================

# Abre una conexión con la base de datos rosterdb.sqlite.
# Si el archivo no existe, SQLite lo crea automáticamente.
conn = sqlite3.connect('rosterdb.sqlite')

# Crea un cursor para ejecutar instrucciones SQL.
cur = conn.cursor()


# ============================================================
# 2. PREPARAR LA BASE DE DATOS
# ============================================================

# executescript() permite ejecutar varias instrucciones SQL
# dentro de una misma cadena de texto.

cur.executescript('''
DROP TABLE IF EXISTS User;
DROP TABLE IF EXISTS Member;
DROP TABLE IF EXISTS Course;

CREATE TABLE User (
    id     INTEGER PRIMARY KEY,
    name   TEXT UNIQUE
);

CREATE TABLE Course (
    id     INTEGER PRIMARY KEY,
    title  TEXT UNIQUE
);

CREATE TABLE Member (
    user_id     INTEGER,
    course_id   INTEGER,
    role        INTEGER,
    PRIMARY KEY (user_id, course_id)
)
''')


# ============================================================
# 3. SOLICITAR Y LEER EL ARCHIVO JSON
# ============================================================

# Pide al usuario el nombre del archivo que contiene los datos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utiliza
# el archivo roster_data.json.
if len(fname) < 1:
    fname = 'roster_data.json'

# Abre el archivo, lee todo su contenido y lo guarda como texto.
str_data = open(fname).read()

# Convierte el texto JSON en una estructura de datos de Python,
# normalmente una lista de listas en este ejercicio.
json_data = json.loads(str_data)

# Ejemplo de los datos contenidos en el archivo JSON:
# [
#     ["Charley", "si110", 1],
#     ["Mea", "si110", 0]
# ]
#
# Cada elemento contiene:
# [nombre del usuario, título del curso, rol]


# ============================================================
# 4. RECORRER LOS DATOS DEL ARCHIVO JSON
# ============================================================

# Recorre cada registro de la lista json_data.
for entry in json_data:

    # Extrae los datos de cada registro según su posición.
    name = entry[0]   # Nombre del usuario.
    title = entry[1]  # Título del curso.
    role = entry[2]   # Rol del usuario en ese curso.

    # Esta instrucción, si se descomenta, muestra los datos.
    # print((name, title, role))


    # ========================================================
    # 5. INSERTAR EL USUARIO Y OBTENER SU ID
    # ========================================================

    # Inserta el usuario si su nombre todavía no existe.
    # INSERT OR IGNORE evita errores por nombres duplicados,
    # debido a la restricción UNIQUE de la columna name.
    cur.execute('''INSERT OR IGNORE INTO User (name)
        VALUES ( ? )''', (name, ))

    # Busca el ID del usuario mediante su nombre.
    # El signo ? se reemplaza por el valor de name.
    cur.execute('SELECT id FROM User WHERE name = ? ', (name, ))

    # Recupera la fila del resultado y obtiene su primer valor:
    # el ID del usuario.
    user_id = cur.fetchone()[0]


    # ========================================================
    # 6. INSERTAR EL CURSO Y OBTENER SU ID
    # ========================================================

    # Inserta el curso si su título todavía no existe.
    # INSERT OR IGNORE evita insertar títulos duplicados.
    cur.execute('''INSERT OR IGNORE INTO Course (title)
        VALUES ( ? )''', (title, ))

    # Busca el ID del curso mediante su título.
    cur.execute('SELECT id FROM Course WHERE title = ? ', (title, ))

    # Recupera el ID del curso encontrado.
    course_id = cur.fetchone()[0]


    # ========================================================
    # 7. REGISTRAR LA RELACIÓN ENTRE USUARIO Y CURSO
    # ========================================================

    # Inserta o reemplaza la relación entre el usuario y el curso.
    # Se almacenan el ID del usuario, el ID del curso y su rol.
    #
    # INSERT OR REPLACE reemplaza el registro existente si ya
    # hay una fila con la misma clave primaria compuesta:
    # (user_id, course_id).
    cur.execute('''INSERT OR REPLACE INTO Member
        (user_id, course_id, role) VALUES ( ?, ?, ? )''',
        (user_id, course_id, role))


# ============================================================
# 8. GUARDAR LOS CAMBIOS
# ============================================================

# Confirma y guarda todas las modificaciones realizadas
# en la base de datos.
conn.commit()

# Cierra la conexión con la base de datos.
conn.close()

Enter file name: 


In [9]:
import sqlite3

# 1. Conectarse a la base de datos
conn = sqlite3.connect('rosterdb.sqlite')
cur = conn.cursor()

# 2. Ejecutar la consulta SQL
cur.execute('''
    SELECT User.name, Course.title, Member.role
    FROM User
    JOIN Member ON User.id = Member.user_id
    JOIN Course ON Member.course_id = Course.id
    ORDER BY User.name DESC,
             Course.title DESC,
             Member.role DESC
    LIMIT 2
''')

# 3. Recuperar los resultados
rows = cur.fetchall()

# 4. Mostrar los resultados
print("Usuario | Curso | Rol")
print("-" * 50)

for row in rows:
    print(row)

# 5. Cerrar la conexión
conn.close()

Usuario | Curso | Rol
--------------------------------------------------
('Zunera', 'si106', 0)
('Zulaikha', 'si422', 0)


In [10]:
import sqlite3

# 1. Conectarse a la base de datos
conn = sqlite3.connect('rosterdb.sqlite')
cur = conn.cursor()

# 2. Ejecutar la consulta SQL
cur.execute('''
    SELECT 'XYZZY' || hex(User.name || Course.title || Member.role ) AS X FROM 
    User JOIN Member JOIN Course 
    ON User.id = Member.user_id AND Member.course_id = Course.id
    ORDER BY X LIMIT 1;
''')

# 3. Recuperar los resultados
print(cur.fetchall())

# 4. Cerrar la conexión
conn.close()

[('XYZZY4161697661736934333030',)]
